<a href="https://colab.research.google.com/github/Abilaash17/Analysing-IOD-and-El-Nino-impact-in-Sri-Lanka-s-Rainfall/blob/main/IOD_ENSO_SriLanka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENSO, the Indian Ocean Dipole and Sri Lankan Rainfall — Continuation Study (1996–2025)
### Sri Lanka only · IOD and ENSO influence · IOD–ENSO relationship · cyclones · outlook to mid-2027

Continuation of the India/Sri Lanka ENSO study. Questions:
1. How does the **Indian Ocean Dipole (IOD)** relate to Sri Lankan rainfall, alone and alongside ENSO?
2. How strongly are **IOD and ENSO themselves related**? (This matters: correlated predictors cannot be read independently.)
3. Do ENSO/IOD relate to **North Indian Ocean cyclones**, especially those near Sri Lanka?
4. What can be said, *with honest uncertainty*, about **early 2027**?

**Cells tagged `net`** need internet access. Everything else only needs the files those cells create.

**Scope note on forecasting.** Google's WeatherNext models are medium-range (≈15 days) and cannot
forecast 2027. This notebook's outlook is a statistical, cross-validated model driven by
*scenario* ENSO/IOD values; the appendix shows where a short-range AI forecast would fit.

## 0. Setup

In [ ]:
#!pip install --quiet statsmodels xarray netCDF4 seaborn scipy
import os, re, glob, time, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import StringIO
import xarray as xr
from scipy import stats
from scipy.stats import pearsonr, fisher_exact, spearmanr
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass

START, END = 1996, 2025                      # analysis period
CLIM_BASE = ('1991-01-01', '2020-12-31')     # baseline used for SST (DMI) anomalies
SL_LAT, SL_LON = 7.8, 80.7                   # approximate centre of Sri Lanka
BOXES = {'west': (-10, 10, 50, 70), 'se': (-10, 0, 90, 110)}   # DMI boxes: lat0, lat1, lon0, lon1
WIND_MIN = 34                                # kt; cyclonic-storm threshold. Try 25 as a sensitivity check (depressions also bring rain)


## 1. ENSO — Oceanic Niño Index (ONI)

Print the tail of the **raw** file *before* filtering. (In the first study a stale-data diagnosis was
wrong because it was run on an already-filtered table.)

In [ ]:
url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
resp = requests.get(url, timeout=30)
resp.raise_for_status()
oni_raw = pd.read_csv(StringIO(resp.text), sep=r'\s+')
print(oni_raw.tail(4))


In [ ]:
season_to_month = {'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,
                   'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12}
oni_raw['month'] = oni_raw['SEAS'].map(season_to_month)
oni_raw['date'] = pd.to_datetime(dict(year=oni_raw['YR'], month=oni_raw['month'], day=1))
oni_all = (oni_raw.rename(columns={'ANOM': 'ONI'})[['date', 'ONI']]
           .sort_values('date').reset_index(drop=True))

oni = oni_all[(oni_all.date >= f'{START}-01-01') & (oni_all.date <= f'{END}-12-31')].reset_index(drop=True)
oni_2026 = oni_all[oni_all.date > f'{END}-12-31']          # current event, kept OUT of the fits
assert len(oni) == (END - START + 1) * 12, "ONI months missing"

print("Latest ONI value in file:", oni_all.iloc[-1]['date'].strftime('%b %Y'), round(oni_all.iloc[-1]['ONI'], 2))
print("\n2026 ONI values available:")
print(oni_2026.tail(8).to_string(index=False))


## 2. Indian Ocean Dipole — Dipole Mode Index (DMI)

**Definition.** DMI = SST anomaly in the western tropical Indian Ocean (50–70°E, 10°S–10°N) minus the
anomaly in the south-eastern box (90–110°E, 10°S–0°). Positive = warm west / cool east (positive IOD).

**Computed from OISST v2.1** via ERDDAP (the route that worked earlier), anomalies relative to
1991–2020. To keep downloads small the grid is sub-sampled every 1° (stride 4 on the 0.25° grid) and
area-averaged with cos(latitude) weights. *Alternative:* upload NOAA PSL's HadISST-based DMI file
(Option B) and cross-check.

Event definition used here: |DMI| ≥ 0.4 °C for at least 3 consecutive months.

In [ ]:
ERDDAP = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180.nc"

def fetch_box_year(tag, year, tries=4):
    lat0, lat1, lon0, lon1 = BOXES[tag]
    fn = f"oisst_{tag}_{year}.nc"
    if os.path.exists(fn) and year < datetime.date.today().year:   # current year is partial: always refresh
        return fn
    stop = "last" if year == datetime.date.today().year else f"({year}-12-31)"
    q = (f"?sst[({year}-01-01):1:{stop}][(0.0):1:(0.0)]"
         f"[({lat0}):4:({lat1})][({lon0}):4:({lon1})]")
    last = None
    for attempt in range(tries):                                   # ERDDAP 503s are usually transient
        try:
            r = requests.get(ERDDAP + q, timeout=300)
            r.raise_for_status()
            if r.content[:3] != b'CDF' and r.content[1:4] != b'HDF':   # ERDDAP returns HTML on errors
                raise ValueError(f"{tag} {year}: not a NetCDF file -> {r.text[:200]}")
            with open(fn, 'wb') as f:
                f.write(r.content)
            return fn
        except Exception as e:
            last = e
            time.sleep(5 * (attempt + 1))
    raise last

failures = []
for tag in BOXES:
    for yr in range(1991, datetime.date.today().year + 1):
        try:
            fetch_box_year(tag, yr)
        except Exception as e:
            failures.append((tag, yr, str(e)[:120]))
print("Failures:", failures if failures else "none")


In [ ]:
def box_monthly(tag):
    parts = []
    for fn in sorted(glob.glob(f"oisst_{tag}_*.nc")):
        da = xr.open_dataset(fn)['sst']
        if 'zlev' in da.dims:
            da = da.squeeze('zlev', drop=True)
        w = np.cos(np.deg2rad(da['latitude']))
        parts.append(da.weighted(w).mean(dim=['latitude', 'longitude']).to_series())
    s = pd.concat(parts)
    s.index = pd.to_datetime(s.index)
    return s.sort_index().resample('MS').mean()

def monthly_anom(s):
    base = s[CLIM_BASE[0]:CLIM_BASE[1]]
    clim = base.groupby(base.index.month).mean()
    return s - clim.reindex(s.index.month).values

dmi_all = (monthly_anom(box_monthly('west')) - monthly_anom(box_monthly('se'))).rename('DMI').reset_index()
dmi_all.columns = ['date', 'DMI']
dmi = dmi_all[(dmi_all.date >= f'{START}-01-01') & (dmi_all.date <= f'{END}-12-31')].reset_index(drop=True)
dmi_2026 = dmi_all[dmi_all.date > f'{END}-12-31']

assert len(dmi) == (END - START + 1) * 12, f"DMI has {len(dmi)} months, expected {(END-START+1)*12}"
files_per_box = {t: len(glob.glob(f"oisst_{t}_*.nc")) for t in BOXES}
print("SST files per box (expect", datetime.date.today().year - 1990, "):", files_per_box)
miss = dmi[dmi['DMI'].isna()]
if len(miss):
    print(f"WARNING: DMI missing for {len(miss)} months in years {sorted(set(miss.date.dt.year))}. "
          "Re-run the download cell (it only fetches missing files), then re-run this cell and everything below.")
dmi_all.to_csv('dmi_monthly.csv', index=False)          # checkpoint
print(dmi.describe().round(2))
print("\n2026 DMI (own computation; 'Final' OISST lags the present):")
print(dmi_2026.tail(6).to_string(index=False))


**Option B / validation (optional).** Upload NOAA PSL's DMI file (e.g. `dmi.had.long.data`) and compare.
The parser ignores header/footer lines and treats values ≤ −90 as missing.

In [ ]:
def read_psl_timeseries(path):
    rows = []
    for line in open(path):
        parts = line.split()
        if len(parts) == 13 and re.fullmatch(r'\d{4}', parts[0]):
            vals = [float(v) for v in parts[1:]]
            for m, v in enumerate(vals, start=1):
                rows.append((pd.Timestamp(int(parts[0]), m, 1), np.nan if v <= -90 else v))
    return pd.DataFrame(rows, columns=['date', 'DMI_psl'])

if os.path.exists('dmi.had.long.data'):
    psl = read_psl_timeseries('dmi.had.long.data')
    chk = dmi.merge(psl, on='date').dropna()
    print(f"Own DMI vs PSL DMI: r = {chk['DMI'].corr(chk['DMI_psl']):.3f} over {len(chk)} months")
else:
    print("No PSL file uploaded - skipping cross-check (optional).")


### 2b. IOD and ENSO events, 1996–2025

In [ ]:
def find_events(df, col, thresh, min_len=3, positive=True):
    s = (df[col] >= thresh) if positive else (df[col] <= thresh)
    grp = (s != s.shift()).cumsum()
    out = []
    for _, g in df[s].groupby(grp[s]):
        if len(g) >= min_len:
            idx = g[col].idxmax() if positive else g[col].idxmin()
            out.append({'start': g['date'].min(), 'end': g['date'].max(), 'months': len(g),
                        'peak': round(g.loc[idx, col], 2), 'peak_date': g.loc[idx, 'date']})
    return pd.DataFrame(out)

pIOD = find_events(dmi, 'DMI', 0.4, 3, True)
nIOD = find_events(dmi, 'DMI', -0.4, 3, False)
elnino = find_events(oni, 'ONI', 0.5, 5, True)
lanina = find_events(oni, 'ONI', -0.5, 5, False)
print("Positive IOD events:\n", pIOD[['start', 'end', 'months', 'peak']].to_string(index=False))
print("\nNegative IOD events:\n", nIOD[['start', 'end', 'months', 'peak']].to_string(index=False))
print("\nEl Nino events:\n", elnino[['start', 'end', 'months', 'peak']].to_string(index=False))


In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
a1.bar(oni['date'], oni['ONI'], width=25, color=np.where(oni['ONI'] >= 0.5, 'red', np.where(oni['ONI'] <= -0.5, 'blue', 'gray')))
a1.axhline(0.5, color='red', ls='--', lw=0.8); a1.axhline(-0.5, color='blue', ls='--', lw=0.8)
a1.set_ylabel('ONI (°C)'); a1.set_title('ENSO: Oceanic Niño Index')
a2.bar(dmi['date'], dmi['DMI'], width=25, color=np.where(dmi['DMI'] >= 0.4, 'darkorange', np.where(dmi['DMI'] <= -0.4, 'teal', 'gray')))
a2.axhline(0.4, color='darkorange', ls='--', lw=0.8); a2.axhline(-0.4, color='teal', ls='--', lw=0.8)
a2.set_ylabel('DMI (°C)'); a2.set_title('IOD: Dipole Mode Index (orange = positive, teal = negative)')
plt.tight_layout()
plt.savefig('oni_dmi_timeline.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()


## 3. How are the IOD and ENSO related?

Two caveats up front: (i) monthly series are autocorrelated, so naive p-values overstate significance — an
**effective sample size** (Bretherton et al. style) is used for the all-month lag correlations;
(ii) positive IOD events often co-occur with El Niño, so any model with both predictors faces
**collinearity**.

In [ ]:
def pearson_neff(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    r, p = pearsonr(x, y)
    r1x, r1y = pd.Series(x).autocorr(1), pd.Series(y).autocorr(1)
    n = len(x)
    n_eff = min(max(n * (1 - r1x * r1y) / (1 + r1x * r1y), 3.5), n)
    t = r * np.sqrt((n_eff - 2) / (1 - r ** 2))
    return r, p, n_eff, 2 * stats.t.sf(abs(t), n_eff - 2)

ind = oni.merge(dmi, on='date').reset_index(drop=True)   # keep NaN months so lags stay aligned

rows = []
for lag in range(-6, 7):                      # lag > 0: ONI leads DMI by `lag` months
    pair = pd.concat([ind['ONI'].shift(lag), ind['DMI']], axis=1).dropna()
    r, p, ne, pe = pearson_neff(pair.iloc[:, 0], pair.iloc[:, 1])
    rows.append({'lag_months(ONI leads +)': lag, 'r': round(r, 3), 'n_eff': round(ne, 1), 'p_naive': p, 'p_eff': pe})
lagdf = pd.DataFrame(rows)
print(lagdf.to_string(index=False))

bym = []
for m in range(1, 13):
    sub = ind[ind.date.dt.month == m].dropna()
    r, p = pearsonr(sub['ONI'], sub['DMI'])
    bym.append((m, r, p))
bym = pd.DataFrame(bym, columns=['month', 'r', 'p'])
plt.figure(figsize=(10, 4))
plt.bar(bym['month'], bym['r'], color=np.where(bym['p'] < 0.05, 'firebrick', 'lightgray'), edgecolor='black')
plt.axhline(0, color='k', lw=0.8); plt.xticks(range(1, 13))
plt.xlabel('Calendar month'); plt.ylabel('Pearson r (ONI vs DMI)')
plt.title('ONI–DMI correlation by calendar month (red = p<0.05, n=30 years each)')
plt.tight_layout(); plt.savefig('oni_dmi_by_month.png', dpi=300, bbox_inches='tight', facecolor='white'); plt.show()


In [ ]:
# Year-level classification, using Sep–Nov means (when the IOD peaks)
son = (ind[ind.date.dt.month.isin([9, 10, 11])]
       .groupby(ind.date.dt.year)[['ONI', 'DMI']].mean()).dropna()     # years with missing DMI are dropped, not misclassified
son['ENSO'] = np.where(son['ONI'] >= 0.5, 'El Nino', np.where(son['ONI'] <= -0.5, 'La Nina', 'Neutral'))
son['IOD'] = np.where(son['DMI'] >= 0.4, 'pIOD', np.where(son['DMI'] <= -0.4, 'nIOD', 'Neutral IOD'))

print(pd.crosstab(son['ENSO'], son['IOD'], margins=True))
for e in ['El Nino', 'La Nina']:
    for i in ['pIOD', 'nIOD']:
        yrs = son.index[(son.ENSO == e) & (son.IOD == i)].tolist()
        print(f"{e:8s} + {i}: {yrs}")

tab = [[((son.ENSO == 'El Nino') & (son.IOD == 'pIOD')).sum(), ((son.ENSO == 'El Nino') & (son.IOD != 'pIOD')).sum()],
       [((son.ENSO != 'El Nino') & (son.IOD == 'pIOD')).sum(), ((son.ENSO != 'El Nino') & (son.IOD != 'pIOD')).sum()]]
odds, pf = fisher_exact(tab)
print(f"\nFisher exact (El Nino vs positive IOD): odds ratio = {odds:.2f}, p = {pf:.4f}")
print("SON-mean ONI vs DMI across years: r = %.3f (p = %.4f)" % pearsonr(son['ONI'], son['DMI']))


## 4. Sri Lankan rainfall: ENSO and IOD, separately and with each other removed

Uses the zone-averaged monthly CHIRPS checkpoint from the first study (`precip_zone_checkpoint.csv`;
upload it, no re-download needed). For each of 12 months × 2 zones × 2 predictors we report the simple
correlation and the **partial correlation** (the other index held constant). A **Benjamini–Hochberg
q-value** controls the false-discovery rate across the whole family; `•` in the heatmaps marks q < 0.10.

In [ ]:
# This notebook runs in its own Colab runtime, so the checkpoint from the first study is not here yet.
CKPT = 'precip_zone_checkpoint.csv'
if not os.path.exists(CKPT):
    try:
        from google.colab import files
        print("Select precip_zone_checkpoint.csv from your computer...")
        up = files.upload()
        for name in up:                              # Colab renames duplicates, e.g. 'name (1).csv'
            if name.startswith('precip_zone_checkpoint') and name != CKPT:
                os.replace(name, CKPT)
    except ImportError:
        raise FileNotFoundError(f"{CKPT} not found - place it next to the notebook.")
if not os.path.exists(CKPT):
    raise FileNotFoundError(f"{CKPT} still missing - re-upload it (it was saved by the first study's Section 4).")

precip = pd.read_csv(CKPT, parse_dates=['date'])
missing = {'date', 'srilanka_wet', 'srilanka_dry'} - set(precip.columns)
assert not missing, f"Checkpoint is missing columns: {missing}"
precip = precip[(precip.date >= f'{START}-01-01') & (precip.date <= f'{END}-12-31')]
print(precip.shape, precip['date'].min().date(), '->', precip['date'].max().date())

def monthly_anomaly(df, col):
    d = df[['date', col]].rename(columns={col: 'val'}).copy()
    d['m'] = d['date'].dt.month
    d['anom'] = d['val'] - d.groupby('m')['val'].transform('mean')
    return d[['date', 'anom']]

zones = {'Sri Lanka Wet (SW)': 'srilanka_wet', 'Sri Lanka Dry (N/E)': 'srilanka_dry'}
panel = {name: monthly_anomaly(precip, col).merge(ind[['date', 'ONI', 'DMI']], on='date').dropna()
         for name, col in zones.items()}
print({k: v.shape for k, v in panel.items()})


In [ ]:
def partial_corr(x, y, z):
    x, y, z = (np.asarray(a, float) for a in (x, y, z))
    Z = np.column_stack([np.ones(len(z)), z])
    rx = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
    ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    r = np.corrcoef(rx, ry)[0, 1]
    dfree = len(x) - 3
    t = r * np.sqrt(dfree / (1 - r ** 2))
    return r, 2 * stats.t.sf(abs(t), dfree)

def bh_qvalues(p):
    p = np.asarray(p, float); n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(q, 1)
    return out

rows = []
for zname, d in panel.items():
    for m in range(1, 13):
        s = d[d.date.dt.month == m]
        for pred, other in (('ONI', 'DMI'), ('DMI', 'ONI')):
            r, p = pearsonr(s[pred], s['anom'])
            pr, pp = partial_corr(s[pred], s['anom'], s[other])
            rows.append({'zone': zname, 'month': m, 'predictor': pred, 'n': len(s),
                         'r': r, 'p': p, 'partial_r': pr, 'partial_p': pp})
sweep = pd.DataFrame(rows)
sweep['q'] = bh_qvalues(sweep['p'])
sweep['partial_q'] = bh_qvalues(sweep['partial_p'])
print("Tests per family:", len(sweep), "| expected false positives at p<0.05 by chance:", round(0.05 * len(sweep), 1))
print(sweep[(sweep.p < 0.05) | (sweep.partial_p < 0.05)]
      .round(3).sort_values('p').to_string(index=False))


In [ ]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

def heat(rcol, pcol, qcol, title, fname):
    sweep['row'] = sweep['zone'] + ' | ' + sweep['predictor']
    R = sweep.pivot(index='row', columns='month', values=rcol)
    P = sweep.pivot(index='row', columns='month', values=pcol)
    Q = sweep.pivot(index='row', columns='month', values=qcol)
    fig, ax = plt.subplots(figsize=(14, 4.2))
    im = ax.imshow(R.values, cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='auto')
    for i in range(R.shape[0]):
        for j in range(R.shape[1]):
            dot = '•' if Q.values[i, j] < 0.10 else ''
            ax.text(j, i, f"{R.values[i, j]:+.2f}{stars(P.values[i, j])}{dot}", ha='center', va='center',
                    fontsize=8, color='white' if abs(R.values[i, j]) > 0.35 else 'black')
    ax.set_xticks(range(12)); ax.set_xticklabels(month_names)
    ax.set_yticks(range(R.shape[0])); ax.set_yticklabels(R.index)
    ax.set_title(title); plt.colorbar(im, ax=ax, label='r')
    plt.tight_layout(); plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white'); plt.show()

heat('r', 'p', 'q', 'Simple correlation with Sri Lankan rainfall anomaly  (* p<0.05, ** p<0.01, *** p<0.001, • BH q<0.10)', 'sl_heatmap_simple.png')
heat('partial_r', 'partial_p', 'partial_q', 'Partial correlation (other index held constant)', 'sl_heatmap_partial.png')


## 5. Seasonal models: climatology vs ONI vs DMI vs ONI+DMI

Two windows tied to Sri Lanka's cycle: **OND** (post-monsoon; the current season) and **FMA** (the late-NE-monsoon/dry
spell found to be ENSO-sensitive in the first study). Every model is scored by **leave-one-year-out
cross-validation** against a climatology baseline (MSE skill score; > 0 means it beats climatology).
With n = 30 an in-sample R² is optimistic; cross-validated skill is the honest measure.

In [ ]:
WINDOWS = {'OND': [10, 11, 12], 'FMA': [2, 3, 4]}
MODELS = {'Climatology': [], 'ONI': ['ONI'], 'DMI': ['DMI'], 'ONI+DMI': ['ONI', 'DMI']}

def window_table(zname, months):
    d = panel[zname]
    d = d[d.date.dt.month.isin(months)]
    return d.groupby(d.date.dt.year).agg(anom=('anom', 'mean'), ONI=('ONI', 'mean'), DMI=('DMI', 'mean'))

def loocv_errors(g, cols):
    errs = []
    for i in range(len(g)):
        tr, te = g.drop(g.index[i]), g.iloc[[i]]
        if cols:
            m = sm.OLS(tr['anom'], sm.add_constant(tr[cols])).fit()
            pred = m.predict(sm.add_constant(te[cols], has_constant='add')).values[0]
        else:
            pred = tr['anom'].mean()
        errs.append(te['anom'].values[0] - pred)
    return np.array(errs)

results = []
for zname in zones:
    for wname, months in WINDOWS.items():
        g = window_table(zname, months)
        mse_clim = np.mean(loocv_errors(g, []) ** 2)
        r_x = g[['ONI', 'DMI']].corr().iloc[0, 1]
        for mname, cols in MODELS.items():
            mse = np.mean(loocv_errors(g, cols) ** 2)
            if cols:
                fit = sm.OLS(g['anom'], sm.add_constant(g[cols])).fit()
                r2, pF = fit.rsquared, fit.f_pvalue
                coefs = {c: round(fit.params[c], 3) for c in cols}
                pvals = {c: round(fit.pvalues[c], 4) for c in cols}
            else:
                r2, pF, coefs, pvals = 0.0, np.nan, {}, {}
            results.append({'zone': zname, 'window': wname, 'model': mname, 'R2_in_sample': round(r2, 3),
                            'LOOCV_skill': round(1 - mse / mse_clim, 3), 'coefs': coefs, 'p': pvals,
                            'corr(ONI,DMI)': round(r_x, 2), 'VIF': round(1 / (1 - r_x ** 2), 2)})
res = pd.DataFrame(results)
print(res.to_string(index=False))


### 5b. Analog composites: what happened in past El Niño years *with* and *without* a positive IOD?

Because every positive-IOD year in this sample is also an El Niño year (Section 3), regression cannot separate the
two. Composites can at least show what Sri Lanka got in the six past years that resemble the present
(El Niño **and** positive IOD in Sep–Nov) versus El Niño years without a positive IOD. Group sizes are tiny, so read
these as descriptive. For FMA the preceding Sep–Nov category is used (SON of year Y → FMA of year Y+1).

In [ ]:
from scipy.stats import mannwhitneyu

def analog_table(zname, wname, lag_years):
    g = window_table(zname, WINDOWS[wname])
    cat = son[['ENSO', 'IOD']].copy()
    cat.index = cat.index + lag_years            # SON of year Y -> window of year Y+lag
    d = g.join(cat, how='inner')
    d['group'] = np.where((d.ENSO == 'El Nino') & (d.IOD == 'pIOD'), 'El Nino + pIOD',
                 np.where(d.ENSO == 'El Nino', 'El Nino, no pIOD',
                 np.where(d.ENSO == 'La Nina', 'La Nina', 'Neutral ENSO')))
    return d

for zname in zones:
    for wname, lag in (('OND', 0), ('FMA', 1)):
        d = analog_table(zname, wname, lag)
        print(f"\n=== {zname} | {wname} (mean rainfall anomaly, mm/day) ===")
        summ = d.groupby('group')['anom'].agg(['mean', 'std', 'count']).round(2)
        summ['years'] = d.groupby('group').apply(lambda x: list(x.index))
        print(summ.to_string())
        a = d.loc[d.group == 'El Nino + pIOD', 'anom']
        for other, label in (('El Nino, no pIOD', 'El Nino without pIOD'), (None, 'all other years')):
            b = d.loc[d.group == other, 'anom'] if other else d.loc[d.group != 'El Nino + pIOD', 'anom']
            if len(a) >= 3 and len(b) >= 2:
                u, p = mannwhitneyu(a, b, alternative='two-sided')
                print(f"  Mann-Whitney, El Nino + pIOD vs {label}: p = {p:.3f} (n = {len(a)} vs {len(b)})")


## 6. Cyclones in the North Indian Ocean (IBTrACS)

**Data.** IBTrACS v04r01, North Indian basin (IMD is the WMO agency here). If NCEI is unreachable from
your Colab (it refused connections in the first study), download
`ibtracs.NI.list.v04r01.csv` in a browser and upload it.

**Definitions.** A *cyclonic storm* = maximum wind ≥ 34 kt (WMO/IMD wind, falling back to JTWC wind
only where missing — note the two use different averaging times). Seasons by genesis month:
Apr–Jun (pre-monsoon) and Oct–Dec (post-monsoon). A storm is *Sri Lanka-near* if any track point lies
within 400 km of the island's centre. Counts are small (a few per season), so Poisson methods are used and
conclusions are framed as exploratory.

In [ ]:
IB_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
          "v04r01/access/csv/ibtracs.NI.list.v04r01.csv")
LOCAL_IB = 'ibtracs.NI.list.v04r01.csv'
src = LOCAL_IB if os.path.exists(LOCAL_IB) else IB_URL
ib_raw = pd.read_csv(src, skiprows=[1], low_memory=False)      # row 1 holds units
print(src, ib_raw.shape)


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

ib = ib_raw.copy()
if 'TRACK_TYPE' in ib.columns:
    ib = ib[ib['TRACK_TYPE'].astype(str).str.contains('main', case=False)]
for c in ['LAT', 'LON', 'WMO_WIND', 'USA_WIND']:
    ib[c] = pd.to_numeric(ib[c], errors='coerce')
ib['time'] = pd.to_datetime(ib['ISO_TIME'], errors='coerce')
ib = ib.dropna(subset=['time', 'LAT', 'LON']).sort_values('time')
ib['wind'] = ib['WMO_WIND'].fillna(ib['USA_WIND'])
ib['dist_sl'] = haversine(ib['LAT'], ib['LON'], SL_LAT, SL_LON)

agg = dict(name=('NAME', 'first'), gen_time=('time', 'first'), gen_lat=('LAT', 'first'),
           gen_lon=('LON', 'first'), max_wind=('wind', 'max'), min_dist_sl=('dist_sl', 'min'))
if 'SUBBASIN' in ib.columns:
    agg['subbasin'] = ('SUBBASIN', 'first')
storms = ib.groupby('SID').agg(**agg).reset_index()
storms['year'] = storms['gen_time'].dt.year
storms['month'] = storms['gen_time'].dt.month
storms = storms[(storms.year >= START) & (storms.year <= END) & (storms.max_wind >= WIND_MIN)].copy()
storms['season'] = np.where(storms.month.isin([10, 11, 12]), 'OND', np.where(storms.month.isin([4, 5, 6]), 'AMJ', 'other'))
storms['near_SL'] = storms['min_dist_sl'] <= 400

years = pd.Index(range(START, END + 1), name='year')
cy = pd.DataFrame(index=years)
cy['n_all'] = storms.groupby('year').size()
cy['n_OND'] = storms[storms.season == 'OND'].groupby('year').size()
cy['n_AMJ'] = storms[storms.season == 'AMJ'].groupby('year').size()
cy['n_nearSL'] = storms[storms.near_SL].groupby('year').size()
cy = cy.fillna(0).astype(int)
print(f"{len(storms)} storms with max wind >= {WIND_MIN} kt, {START}-{END}; mean per year = {cy['n_all'].mean():.2f}")
print(cy.describe().loc[['mean', 'std', 'max']].round(2))


In [ ]:
cyd = cy.join(son[['ONI', 'DMI', 'ENSO', 'IOD']], how='inner')

def poisson_fit(ycol, label):
    X = sm.add_constant(cyd[['ONI', 'DMI']])
    m = sm.GLM(cyd[ycol], X, family=sm.families.Poisson()).fit(cov_type='HC0')
    disp = m.pearson_chi2 / m.df_resid
    print(f"\n--- Poisson GLM: {label} ~ ONI + DMI (Sep–Nov means; robust SE) ---")
    print(m.summary().tables[1])
    print(f"Overdispersion (Pearson chi2/df) = {disp:.2f}  (>1.5 means treat p-values cautiously)")

for col, label in [('n_OND', 'post-monsoon (Oct–Dec) storms'), ('n_nearSL', 'storms passing within 400 km of Sri Lanka')]:
    poisson_fit(col, label)

print("\nMean storm counts by category (n = number of years in group):")
for key in ['ENSO', 'IOD']:
    print(cyd.groupby(key)[['n_OND', 'n_nearSL']].agg(['mean', 'count']).round(2), "\n")


In [ ]:
# Location and intensity of post-monsoon storms vs. the indices
sto = storms[storms.season == 'OND'].merge(son[['ONI', 'DMI']], left_on='year', right_index=True)
print(f"{len(sto)} post-monsoon storms")
for var in ['gen_lat', 'gen_lon', 'max_wind']:
    for idx in ['ONI', 'DMI']:
        rho, p = spearmanr(sto[idx], sto[var])
        print(f"Spearman {var:9s} vs {idx}: rho = {rho:+.2f}, p = {p:.3f}")

# Storms from the same year share one ENSO/IOD value, so storm-level p-values overstate the evidence.
yr = (sto.groupby('year').agg(gen_lon=('gen_lon', 'mean'), gen_lat=('gen_lat', 'mean'),
                              max_wind=('max_wind', 'max')).join(son[['ONI', 'DMI']], how='inner'))
print(f"\nYear-level tests (one value per year, n = {len(yr)}):")
for var in ['gen_lon', 'gen_lat', 'max_wind']:
    for idx in ['ONI', 'DMI']:
        rho, p = spearmanr(yr[idx], yr[var])
        print(f"  Spearman mean-{var:8s} vs {idx}: rho = {rho:+.2f}, p = {p:.3f}")

if 'subbasin' in sto.columns:
    print("\nPost-monsoon storms by sub-basin:\n", sto['subbasin'].value_counts().to_string())
    for sb, grp in sto.groupby('subbasin'):
        if len(grp) >= 15:
            for idx in ['ONI', 'DMI']:
                rho, p = spearmanr(grp[idx], grp['gen_lon'])
                print(f"  within {sb}: gen_lon vs {idx}: rho = {rho:+.2f}, p = {p:.3f} (n = {len(grp)} storms)")
    share = (sto.assign(is_AS=(sto['subbasin'] == 'AS')).groupby('year')['is_AS'].mean()
             .rename('share_AS').to_frame().join(son[['ONI', 'DMI']], how='inner'))
    for idx in ['ONI', 'DMI']:
        rho, p = spearmanr(share[idx], share['share_AS'])
        print(f"  Arabian Sea share of post-monsoon storms vs {idx}: rho = {rho:+.2f}, p = {p:.3f} (n = {len(share)} years)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=cyd, x='ENSO', y='n_OND', ax=ax[0], order=['La Nina', 'Neutral', 'El Nino'])
ax[0].set_title('Post-monsoon storms per year by ENSO phase (Sep–Nov)')
sns.boxplot(data=cyd, x='IOD', y='n_OND', ax=ax[1], order=['nIOD', 'Neutral IOD', 'pIOD'])
ax[1].set_title('Post-monsoon storms per year by IOD phase')
plt.tight_layout(); plt.savefig('cyclones_by_phase.png', dpi=300, bbox_inches='tight', facecolor='white'); plt.show()


## 7. Outlook: OND 2026 (in progress) and FMA 2027

**Inputs are scenarios, not forecasts.** Edit them from the latest CPC/IRI/BoM releases. Placeholders
reflect conditions at the time of writing: observed ONI reached +2.46 (ASO 2026) and this notebook's own DMI was +0.62 (Sep) and +0.84 (Oct 2026), fading to neutral by January 2027. **Keep DMI inputs on this notebook's own scale** (IRI/BoM quote slightly different values because of different baselines and grids).

Every prediction shows (a) the raw value, (b) the value with inputs **capped at the observed 1996–2025
range** (to avoid extrapolation, the error that produced an implausible +11.8 mm/day last time), and (c) a 95 %
**prediction interval**. If a model's cross-validated skill is ≤ 0, its prediction should not be emphasised.

In [ ]:
SCENARIOS = {
    'OND': {'label': 'OND 2026 (in progress)',
            'cases': {'ONI 2.0 / DMI +0.7': {'ONI': 2.0, 'DMI': 0.7},
                      'ONI 2.5 / DMI +0.7': {'ONI': 2.5, 'DMI': 0.7},
                      'ONI 3.0 / DMI +0.7': {'ONI': 3.0, 'DMI': 0.7}}},
    'FMA': {'label': 'FMA 2027',
            'cases': {'ONI 1.0 / DMI 0.0': {'ONI': 1.0, 'DMI': 0.0},
                      'ONI 1.5 / DMI 0.0': {'ONI': 1.5, 'DMI': 0.0},
                      'ONI 2.0 / DMI 0.0': {'ONI': 2.0, 'DMI': 0.0}}},
}

def predict_pi(g, cols, vals):
    if not cols:
        n, mu, s = len(g), g['anom'].mean(), g['anom'].std(ddof=1)
        h = stats.t.ppf(0.975, n - 1) * s * np.sqrt(1 + 1 / n)
        return mu, mu - h, mu + h
    m = sm.OLS(g['anom'], sm.add_constant(g[cols])).fit()
    Xn = sm.add_constant(pd.DataFrame([{c: vals[c] for c in cols}]), has_constant='add')
    f = m.get_prediction(Xn).summary_frame(alpha=0.05)
    return f['mean'].iloc[0], f['obs_ci_lower'].iloc[0], f['obs_ci_upper'].iloc[0]

out = []
for zname in zones:
    for wname, sc in SCENARIOS.items():
        g = window_table(zname, WINDOWS[wname])
        rng = {c: (g[c].min(), g[c].max()) for c in ['ONI', 'DMI']}
        for case, vals in sc['cases'].items():
            capped = {c: float(np.clip(vals[c], *rng[c])) for c in vals}
            for mname, cols in MODELS.items():
                if not cols:
                    continue
                was_capped = any(abs(capped[c] - vals[c]) > 1e-9 for c in cols)   # only inputs this model uses
                raw, _, _ = predict_pi(g, cols, vals)
                cap, lo, hi = predict_pi(g, cols, capped)
                skill = res[(res.zone == zname) & (res.window == wname) & (res.model == mname)]['LOOCV_skill'].iloc[0]
                out.append({'zone': zname, 'window': sc['label'], 'scenario': case, 'model': mname,
                            'raw_mm/day': round(raw, 2), 'capped_mm/day': round(cap, 2),
                            '95%PI': f"[{lo:+.2f}, {hi:+.2f}]", 'capped?': was_capped, 'LOOCV_skill': skill})
outlook = pd.DataFrame(out)
outlook['beats_climatology'] = outlook['LOOCV_skill'] > 0
print(outlook.to_string(index=False))
outlook.to_csv('sl_outlook_scenarios.csv', index=False)


## Interpretation guardrails (read before writing anything up)

- **Collinearity.** ONI and DMI are correlated (Section 3). Use partial correlations and cross-validated skill, not the sign of one coefficient.
- **Multiple testing.** 48 tests per family; q-values (`•`) are the safer evidence than raw stars.
- **Small samples.** n = 30 years per month; cyclone counts are a handful per year. Describe cyclone results as exploratory.
- **Scenarios.** Outlook numbers depend entirely on the scenario inputs; they are not a meteorological forecast.
- **Horizon.** ENSO forecasts reach ~mid-2027 and the IOD returns to neutral by early 2027, so nothing here supports statements about the second half of 2027.
- **Cross-check.** Compare against dynamical seasonal forecasts (NMME, IRI, national met-service outlooks) before drawing conclusions.